# Project Review 1: EDA & Statistical Tests (Complex Dataset)

This notebook is specifically structured to fulfill the 10-mark rubric requirements for **Review 1**, utilizing the massive 370-client UCI Electricity Load Diagrams dataset.

## 1. Research Problem & Question (1 Mark)

**Research Problem:** 
Modern power grids require highly accurate load forecasting to optimize energy generation and distribution. However, forecasting becomes immensely complex when dealing with hundreds of diverse consumers, each with unique consumption behaviors, rather than a single aggregated grid.

**Research Question:**
How can we effectively model and forecast the electricity demand of 370 distinct consumers? What are the underlying statistical properties (stationarity, seasonality, autocorrelation) of the grid's aggregated load, and does it require advanced multi-series Deep Learning models (like Transformers) compared to traditional statistical baselines (ARIMA)?

In [ ]:

import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


## 2. Dataset Understanding (1 Mark)

We are utilizing the highly complex **UCI Electricity Load Diagrams 2011-2014** dataset.
*   **Scope:** Electricity consumption of **370 different clients** (industries and households) over 4 years.
*   **Frequency:** Data is recorded every 15 minutes.
*   **Data Engineering Required:** The raw 710MB text file uses semi-colons as separators and commas as decimal points, which must be strictly parsed. Because 370 parallel time-series are computationally heavy, we will aggregate them into a `Total_Load` (Grid Demand) for baseline statistical analysis, resampled to a Daily frequency.

In [2]:
data_path = '../data/LD2011_2014.txt'

if not os.path.exists(data_path):
    print("Error: Data file not found!")
else:
    # Parse the European number format (decimal=',')
    print("Loading massive dataset, please wait...")
    df = pd.read_csv(data_path, sep=';', decimal=',', parse_dates=[0], index_col=0, low_memory=False)
    df.index.name = 'datetime'
    
    # Resample to Daily frequency to make EDA manageable
    df_daily = df.resample('D').sum()
    
    # Create a Total Aggregated Load column
    df_daily['Total_Load'] = df_daily.sum(axis=1)
    
    print(f"\nData loaded! Shape of daily data: {df_daily.shape}")
    display(df_daily[['Total_Load', 'MT_001', 'MT_002', 'MT_370']].head())

Loading massive dataset, please wait...

Data loaded! Shape of daily data: (1462, 371)


,Total_Load,MT_001,MT_002,MT_370
datetime,,,,
2011-01-01,6.852976e+06,0.0,0.0,0.0
2011-01-02,1.114135e+07,0.0,0.0,0.0
2011-01-03,1.124907e+07,0.0,0.0,0.0
2011-01-04,1.146786e+07,0.0,0.0,0.0
2011-01-05,1.152175e+07,0.0,0.0,0.0


## 3. Time-Series Visualization (1 Mark)
We visualize the aggregated grid demand (the sum of all 370 consumers) over the 4-year period using an interactive chart.

In [7]:
if 'df_daily' in locals():
    fig = px.line(df_daily, x=df_daily.index, y='Total_Load', 
                  title='Time-Series Visualization: Total Aggregated Grid Load (370 Clients)', 
                  labels={'Total_Load': 'Total Power (kW)', 'datetime': 'Date'})
    fig.update_xaxes(rangeslider_visible=True)
    fig.show()

## 4. Trend and Seasonality Analysis (2 Marks)
We filter the data starting from 2012 (as 2011 contains sparse data for many clients) and decompose the `Total_Load` to extract the underlying macro Trend and Weekly/Yearly Seasonality.

In [4]:
if 'df_daily' in locals():
    # Filter from 2012 onwards for clean analysis
    ts_target = df_daily['Total_Load']['2012-01-01':]
    
    result = seasonal_decompose(ts_target, model='additive', period=365) # Yearly seasonality
    
    fig_trend = go.Figure()
    fig_trend.add_trace(go.Scatter(x=result.trend.index, y=result.trend.values, mode='lines', line=dict(color='orange')))
    fig_trend.update_layout(title='Decomposition Analysis: Long-Term Trend (2012-2014)', xaxis_title='Date', yaxis_title='Trend')
    fig_trend.show()
    
    # Zoom in to see the strong weekly seasonality (weekends have massive dips)
    slice_2_months = ts_target['2013-01-01':'2013-03-01']
    fig_week = px.line(slice_2_months, title='Decomposition Analysis: Zoomed-in view revealing strict Weekly Seasonality')
    fig_week.show()

## 5. Stationarity Analysis (2 Marks)
For statistical forecasting (ARIMA), the time series must be stationary. We verify this using the **Augmented Dickey-Fuller (ADF) Test**.

*   **H0 (Null Hypothesis):** The series is non-stationary.
*   **H1 (Alternative):** The series is stationary.

Based on our test output:
*   The **RAW Daily Total Load** yielded a p-value of **0.643** (> 0.05). We accept the null hypothesis; the raw data is strongly **NON-STATIONARY**.
*   Applying **1st Order Differencing (d=1)** yielded a p-value of **1.71e-10** (< 0.05). We reject the null hypothesis; the differenced data is robustly **STATIONARY**.

**Conclusion:** Our differencing parameter must be **`d = 1`**.

In [5]:
def perform_adf_test(series, name=""):
    print(f"--- ADF Test: {name} ---")
    result = adfuller(series.dropna())
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.5e}')
    if result[1] <= 0.05:
        print("=> The series is STATIONARY.\n")
    else:
        print("=> The series is NON-STATIONARY.\n")

if 'ts_target' in locals():
    perform_adf_test(ts_target, "RAW Daily Total Load")
    
    print("Applying 1st Order Differencing (d=1) to ensure robust mathematical stationarity...")
    ts_diff = ts_target.diff().dropna()
    perform_adf_test(ts_diff, "DIFFERENCED Total Load (d=1)")

--- ADF Test: RAW Daily Total Load ---
ADF Statistic: -1.2693
p-value: 6.43095e-01
=> The series is NON-STATIONARY.

Applying 1st Order Differencing (d=1) to ensure robust mathematical stationarity...
--- ADF Test: DIFFERENCED Total Load (d=1) ---
ADF Statistic: -7.2576
p-value: 1.71460e-10
=> The series is STATIONARY.



### Visualizing the Stationary (Differenced) Data
Below is the plot of the time series *after* 1st order differencing. Notice how the long-term upward trend has been completely mathematically removed. The data now fluctuates consistently around a constant mean of zero, which is the visual hallmark of strict stationarity.

In [ ]:
if 'ts_diff' in locals():
    fig_diff = px.line(ts_diff, title='Stationary Data (After 1st Order Differencing)', 
                       labels={'value': 'Differenced Power', 'datetime': 'Date'})
    # Add a horizontal line at 0 to show the constant mean
    fig_diff.add_hline(y=0, line_dash="dash", line_color="red")
    fig_diff.update_xaxes(rangeslider_visible=True)
    fig_diff.show()

## 6. ACF/PACF Analysis (2 Marks)
We analyze the Autocorrelation (ACF) and Partial Autocorrelation (PACF) plots on our stationary, differenced data to determine our `p` and `q` parameters.

**Determining `p` (AutoRegressive order from PACF):**
The PACF plot shows a significant negative spike at Lag 1, before dropping into the red confidence interval at Lags 2-5. This justifies setting our standard AR parameter to **`p = 1`**. Furthermore, we observe a massive positive spike exactly at Lag 7, mathematically confirming the 7-day weekly seasonality.

**Determining `q` (Moving Average order from ACF):**
The ACF plot mirrors this behavior, with a sharp negative spike at Lag 1 that quickly drops off into the confidence interval, followed by another massive seasonal spike at Lag 7. The sharp cutoff after the first lag justifies setting our MA parameter to **`q = 1`**.

In [6]:
def plot_interactive_acf_pacf(series, lags=30):
    acf_vals = acf(series, nlags=lags)
    pacf_vals = pacf(series, nlags=lags)
    
    fig_acf = go.Figure()
    fig_acf.add_trace(go.Bar(x=np.arange(len(acf_vals)), y=acf_vals, name='ACF'))
    fig_acf.add_trace(go.Scatter(x=np.arange(len(acf_vals)), y=[1.96/np.sqrt(len(series))]*len(acf_vals), mode='lines', line=dict(color='red', dash='dash')))
    fig_acf.add_trace(go.Scatter(x=np.arange(len(acf_vals)), y=[-1.96/np.sqrt(len(series))]*len(acf_vals), mode='lines', line=dict(color='red', dash='dash')))
    fig_acf.update_layout(title='Autocorrelation Function (ACF) -> Determines q', xaxis_title='Lag', showlegend=False)
    fig_acf.show()
    
    fig_pacf = go.Figure()
    fig_pacf.add_trace(go.Bar(x=np.arange(len(pacf_vals)), y=pacf_vals, name='PACF'))
    fig_pacf.add_trace(go.Scatter(x=np.arange(len(pacf_vals)), y=[1.96/np.sqrt(len(series))]*len(pacf_vals), mode='lines', line=dict(color='red', dash='dash')))
    fig_pacf.add_trace(go.Scatter(x=np.arange(len(pacf_vals)), y=[-1.96/np.sqrt(len(series))]*len(pacf_vals), mode='lines', line=dict(color='red', dash='dash')))
    fig_pacf.update_layout(title='Partial Autocorrelation Function (PACF) -> Determines p', xaxis_title='Lag', showlegend=False)
    fig_pacf.show()

if 'ts_diff' in locals():
    plot_interactive_acf_pacf(ts_diff)

## 7. Initial Findings & Research Hypothesis

**Elaborate Findings on Consumption Behaviors:**
The exploratory data analysis of the UCI Electricity Load Diagrams dataset (2011-2014) has revealed several critical insights regarding the macro and micro consumption behaviors of the 370 monitored clients. By decomposing the aggregated load, we identified a robust, long-term upward trend that initiated in mid-2012, scaling from roughly 20.4M kW to a peak of 21.8M kW by early 2014. This macro-trend indicates overall network growth or increased baseline consumption among the sampled entities. However, the most defining characteristic of this dataset is its intense, rhythmic volatility on a micro scale. 

Zooming into the decomposed data exposed a rigid 7-day weekly seasonality. Power consumption consistently plummets every seventh day, strongly implying that a vast majority of the 370 clients are industrial or commercial enterprises that power down their operations during the weekends. Statistically, this volatility rendered the raw aggregated data highly non-stationary (ADF p-value = 0.643). To mathematically stabilize the mean and variance for baseline modeling, we applied a first-order differencing technique (`d=1`), which successfully annihilated the overarching trend and achieved absolute stationarity (ADF p-value = $1.71 \times 10^{-10}$). The visualization of this differenced data confirms a constant mean oscillating around zero.

Finally, analyzing the Autocorrelation Function (ACF) and Partial Autocorrelation Function (PACF) on this stationary data provided the exact hyperparameters required for modeling. Both the ACF and PACF exhibited significant negative spikes at Lag 1 before falling into the confidence intervals, justifying an AutoRegressive order of `p=1` and a Moving Average order of `q=1`. Furthermore, massive recurring spikes exactly at Lag 7 in both plots mathematically mandate the integration of a seasonal component (with a period of 7) to account for the weekend drops.

**Research Hypothesis:**
We hypothesize that a Seasonal ARIMA (SARIMA) model configured with parameters (1, 1, 1)x(P, D, Q, 7) will serve as a highly effective statistical baseline for forecasting the *aggregated* total grid load. However, we further hypothesize that this statistical approach will fundamentally hit a scalability ceiling when attempting to forecast the distinct, idiosyncratic usage patterns of all 370 consumers independently. To achieve state-of-the-art predictive accuracy across hundreds of multivariate, parallel time-series, a Deep Learning architecture—specifically a Long Short-Term Memory (LSTM) network or a Temporal Fusion Transformer—will be absolutely necessary to dynamically learn the cross-dependencies between clients.

---

### What We Are Going To Do Next (Phase 2 & 3)
1. **Implement Statistical Baselines:** We will code and train the proposed SARIMA(1,1,1) model on the aggregated grid load to establish a concrete performance benchmark (calculating RMSE and MAE).
2. **Transition to Machine Learning:** We will implement tree-based models like XGBoost or LightGBM, utilizing advanced feature engineering (injecting lag features and day-of-week indicators) to see if ML can outperform traditional statistics.
3. **Deploy Deep Learning:** We will build the final LSTM/Transformer architecture designed to ingest the massive 370-dimensional matrix, transitioning our focus from predicting the 'total grid' to forecasting individual consumer demand simultaneously.